# اليوم الثاني — مختبر 3A: تصنيف النصوص
## Day 2 — Lab 3A: Text Classification

**المدربة / Instructor:** ميعاد المري — Meaad Al-Marri  
**المسار:** Core → Explore → Distinction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/almiyead-rgb/bayan-applied-nlp-course/blob/main/notebooks/03_text_classification.ipynb)

> نبني Baseline أولًا، نثبت عدم تسرب المجموعات، ثم ننفذ خطوة ضبط فعلية لـDistilmBERT متعدد اللغات.
>
> Build a baseline, prove group isolation, then perform a real fine-tuning step on multilingual DistilBERT.

**علامة النجاح:** `DAY2_NOTEBOOK3_CORE=PASS`.

## قواعد المختبر

- البيانات اصطناعية ولا تحتوي حالات أو أشخاصًا حقيقيين.
- GPU في Colab Free غير مضمون؛ CPU fallback يجمد المشفر ويدرب task head.
- لا نرفع model weights أو cache إلى GitHub.
- نتائج هذه العينة الصغيرة تسمى `MEASURED_SMOKE`، وليست أداءً إنتاجيًا.
- نفذ **Runtime → Run all**. لا تتجاوز خلية فاشلة.

In [1]:
# تثبيت نسخ اليوم الثاني فقط عند الحاجة
import importlib.metadata
import importlib.util
import subprocess
import sys

REQUIRED = {
    "transformers": "5.15.1",
    "tokenizers": "0.22.2",
    "scikit-learn": "1.9.0",
}
needs_install = []
for distribution, expected in REQUIRED.items():
    try:
        current = importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        current = None
    if current != expected:
        needs_install.append(f"{distribution}=={expected}")

if needs_install:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *needs_install])

if importlib.util.find_spec("torch") is None:
    raise RuntimeError("PyTorch is required. Open this notebook in Google Colab.")

print("Python:", sys.version.split()[0])
print("Environment ready / البيئة جاهزة")

Python: 3.13.15
Environment ready / البيئة جاهزة


In [2]:
import csv
import io
import json
import math
import os
import random
import urllib.request
from collections import Counter
from pathlib import Path

import numpy as np
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC
from torch.optim import AdamW
from transformers import AutoModelForSequenceClassification, AutoTokenizer

os.environ["TOKENIZERS_PARALLELISM"] = "false"
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cuda


## 1) تحميل البيانات وفحصها

يحاول الدفتر قراءة ملف الدورة من GitHub. إذا تعذر، يستخدم عينة اصطناعية مدمجة حتى لا يتوقف Core بسبب رابط البيانات. النموذج نفسه يحتاج تنزيلًا أول مرة.

In [3]:
DATA_URL = "https://raw.githubusercontent.com/almiyead-rgb/bayan-applied-nlp-course/main/data/sample/bayan_day2_classification.csv"
FALLBACK_ROWS = [{"example_id":"F-001","group_id":"DG-A","split":"train","language":"ar","text":"تعذر تسجيل الدخول إلى البوابة","topic":"digital_service","sentiment":"negative"},{"example_id":"F-002","group_id":"DG-A","split":"train","language":"en","text":"I cannot sign in to the portal","topic":"digital_service","sentiment":"negative"},{"example_id":"F-003","group_id":"DG-B","split":"train","language":"ar","text":"الخدمة الإلكترونية سريعة وواضحة","topic":"digital_service","sentiment":"positive"},{"example_id":"F-004","group_id":"DG-B","split":"train","language":"en","text":"The online service is clear and fast","topic":"digital_service","sentiment":"positive"},{"example_id":"F-005","group_id":"PG-A","split":"train","language":"ar","text":"أحتاج معرفة حالة طلب التصريح","topic":"permit","sentiment":"neutral"},{"example_id":"F-006","group_id":"PG-A","split":"train","language":"en","text":"I need the status of my permit request","topic":"permit","sentiment":"neutral"},{"example_id":"F-007","group_id":"PG-B","split":"train","language":"ar","text":"تمت الموافقة على التصريح اليوم","topic":"permit","sentiment":"positive"},{"example_id":"F-008","group_id":"PG-B","split":"train","language":"en","text":"The permit was approved today","topic":"permit","sentiment":"positive"},{"example_id":"F-009","group_id":"HG-A","split":"train","language":"ar","text":"تأخر موعد العيادة هذا الصباح","topic":"health","sentiment":"negative"},{"example_id":"F-010","group_id":"HG-A","split":"train","language":"en","text":"My clinic appointment was delayed","topic":"health","sentiment":"negative"},{"example_id":"F-011","group_id":"HG-B","split":"train","language":"ar","text":"كانت خدمة العيادة ممتازة","topic":"health","sentiment":"positive"},{"example_id":"F-012","group_id":"HG-B","split":"train","language":"en","text":"The clinic service was excellent","topic":"health","sentiment":"positive"},{"example_id":"F-013","group_id":"TG-A","split":"train","language":"ar","text":"الحافلة لم تصل في الوقت المحدد","topic":"transport","sentiment":"negative"},{"example_id":"F-014","group_id":"TG-A","split":"train","language":"en","text":"The bus did not arrive on time","topic":"transport","sentiment":"negative"},{"example_id":"F-015","group_id":"TG-B","split":"train","language":"ar","text":"كانت الرحلة مريحة ومنظمة","topic":"transport","sentiment":"positive"},{"example_id":"F-016","group_id":"TG-B","split":"train","language":"en","text":"The trip was comfortable and organised","topic":"transport","sentiment":"positive"},{"example_id":"F-017","group_id":"DG-V","split":"validation","language":"ar","text":"لم يصل رمز التحقق الرقمي","topic":"digital_service","sentiment":"negative"},{"example_id":"F-018","group_id":"PG-V","split":"validation","language":"en","text":"How can I renew the permit","topic":"permit","sentiment":"neutral"},{"example_id":"F-019","group_id":"HG-V","split":"validation","language":"ar","text":"أحتاج إعادة جدولة الموعد الصحي","topic":"health","sentiment":"neutral"},{"example_id":"F-020","group_id":"TG-V","split":"validation","language":"en","text":"The bus route has changed","topic":"transport","sentiment":"neutral"},{"example_id":"F-021","group_id":"DG-T","split":"test","language":"en","text":"The verification code did not arrive","topic":"digital_service","sentiment":"negative"},{"example_id":"F-022","group_id":"PG-T","split":"test","language":"ar","text":"تأخر إصدار التصريح المطلوب","topic":"permit","sentiment":"negative"},{"example_id":"F-023","group_id":"HG-T","split":"test","language":"en","text":"The clinic appointment was cancelled","topic":"health","sentiment":"negative"},{"example_id":"F-024","group_id":"TG-T","split":"test","language":"ar","text":"توقفت الحافلة قبل المحطة","topic":"transport","sentiment":"negative"}]

try:
    with urllib.request.urlopen(DATA_URL, timeout=20) as response:
        text = response.read().decode("utf-8")
    rows = list(csv.DictReader(io.StringIO(text)))
    DATA_SOURCE = "github_course_file"
except Exception as exc:
    rows = FALLBACK_ROWS
    DATA_SOURCE = f"embedded_fallback:{type(exc).__name__}"

print("Data source:", DATA_SOURCE)
print("Rows:", len(rows))
print("Topics:", Counter(row["topic"] for row in rows))
assert len(rows) >= 24
assert {"ar", "en"} <= {row["language"] for row in rows}

Data source: github_course_file
Rows: 40
Topics: Counter({'digital_service': 10, 'permit': 10, 'health': 10, 'transport': 10})


### عقد رأسي topic وsentiment / Two independent classification heads

يحتوي ملف البيانات على `topic` و`sentiment`. تشغيل المرجع المحفوظ يقيس رأس **topic** كي يبقى smoke محدودًا، لكنه لا يُعد دليلًا لرأس sentiment. في مشروع بيان أعد استخدام split المجمّد نفسه لتدريب رأس sentiment مستقل وlabel map مستقلة، ثم قيّم الرأسين كلًا بمقياسه. تحافظ نسخة fallback كذلك على العمودين حتى لا يتغير العقد عند تعطل GitHub Raw.

In [4]:
def validate_splits(rows):
    required = {"train", "validation", "test"}
    group_owner = {}
    labels_by_split = {name: set() for name in required}
    counts = Counter()
    for row in rows:
        split, group, label = row["split"], row["group_id"], row["topic"]
        if split not in required:
            raise ValueError(f"Unknown split: {split}")
        previous = group_owner.setdefault(group, split)
        if previous != split:
            raise ValueError(f"Group leakage: {group}")
        labels_by_split[split].add(label)
        counts[split] += 1
    all_labels = set().union(*labels_by_split.values())
    for split in required:
        if labels_by_split[split] != all_labels:
            raise ValueError(f"Missing label in {split}")
    return {
        "rows": dict(counts),
        "groups": len(group_owner),
        "labels": sorted(all_labels),
        "group_overlap": 0,
    }

split_report = validate_splits(rows)
print(json.dumps(split_report, ensure_ascii=False, indent=2))
assert split_report["group_overlap"] == 0
print("Split contract=PASS")

{
  "rows": {
    "train": 24,
    "validation": 8,
    "test": 8
  },
  "groups": 20,
  "labels": [
    "digital_service",
    "health",
    "permit",
    "transport"
  ],
  "group_overlap": 0
}
Split contract=PASS


**المتوقع:** كل split يحتوي الفئات الأربع، و`group_overlap` يساوي صفرًا.

## 2) TF-IDF Baseline

نضبط الـbaseline على Train ونقرأ Validation. لا نستخدم Test حتى خلية القياس النهائي بعد تثبيت المسار.

In [5]:
train_rows = [row for row in rows if row["split"] == "train"]
validation_rows = [row for row in rows if row["split"] == "validation"]
test_rows = [row for row in rows if row["split"] == "test"]
LABELS = sorted({row["topic"] for row in rows})
label2id = {label: index for index, label in enumerate(LABELS)}
id2label = {index: label for label, index in label2id.items()}

baseline = make_pipeline(
    TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=1),
    LinearSVC(random_state=SEED),
)
baseline.fit(
    [row["text"] for row in train_rows],
    [row["topic"] for row in train_rows],
)
baseline_val_pred = baseline.predict([row["text"] for row in validation_rows])
baseline_val_f1 = f1_score(
    [row["topic"] for row in validation_rows],
    baseline_val_pred,
    labels=LABELS,
    average="macro",
    zero_division=0,
)
print("Baseline validation macro-F1 (MEASURED_SMOKE):", round(baseline_val_f1, 4))
assert 0.0 <= baseline_val_f1 <= 1.0
print("Baseline=PASS")

Baseline validation macro-F1 (MEASURED_SMOKE): 0.6667
Baseline=PASS


## 3) تنزيل الـcheckpoint وتجهيز النموذج

النموذج عام ومجاني، ولا يحتاج Hugging Face token. عند ظهور warning بأن classification head جديدة فهذا متوقع: المشفر مدرب مسبقًا، أما رأس فئات بيان فيبدأ عشوائيًا.

- GPU: Full fine-tuning قصير.
- CPU: Partial fine-tuning؛ نجمّد معظم المشفر ونحدّث آخر Transformer block مع رأس المهمة.

كلاهما تدريب فعلي، لكن نوعه يسجل في النتائج.

In [6]:
MODEL_ID = "distilbert/distilbert-base-multilingual-cased"
MAX_LENGTH = 64
BATCH_SIZE = 4
NUM_EPOCHS = 2 if DEVICE.type == "cuda" else 12

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID,
        num_labels=len(LABELS),
        label2id=label2id,
        id2label=id2label,
    )
except Exception as exc:
    raise RuntimeError(
        "Checkpoint download failed. Reconnect the runtime and run this cell once. "
        "No API key is required."
    ) from exc

TRAINING_MODE = "full_finetune" if DEVICE.type == "cuda" else "partial_finetune_cpu"
if TRAINING_MODE == "partial_finetune_cpu":
    for parameter in model.base_model.parameters():
        parameter.requires_grad = False
    for parameter in model.base_model.transformer.layer[-1].parameters():
        parameter.requires_grad = True

model.to(DEVICE)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print("Training mode:", TRAINING_MODE)
print("Epochs:", NUM_EPOCHS)
print(f"Trainable parameters: {trainable:,} / {total:,}")
assert trainable > 0

config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  542MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training mode: full_finetune
Epochs: 2
Trainable parameters: 135,327,748 / 135,327,748


In [7]:
def iter_batches(examples, batch_size, *, shuffle=False, seed=SEED):
    indexes = np.arange(len(examples))
    if shuffle:
        np.random.default_rng(seed).shuffle(indexes)
    for start in range(0, len(indexes), batch_size):
        yield [examples[index] for index in indexes[start:start + batch_size]]


def encode_batch(batch):
    encoded = tokenizer(
        [row["text"] for row in batch],
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    encoded["labels"] = torch.tensor(
        [label2id[row["topic"]] for row in batch], dtype=torch.long
    )
    return {key: value.to(DEVICE) for key, value in encoded.items()}


def predict(examples):
    model.eval()
    predictions = []
    with torch.no_grad():
        for batch in iter_batches(examples, BATCH_SIZE):
            encoded = encode_batch(batch)
            logits = model(**encoded).logits
            predictions.extend(logits.argmax(-1).cpu().tolist())
    return [id2label[index] for index in predictions]

print("Batch functions=PASS")

Batch functions=PASS


## 4) خطوة Fine-tuning فعلية

نحدّث أوزانًا داخل آخر Transformer block، ونستخدم validation لاختيار أفضل حقبة، ثم نفتح Test مرة واحدة بعد تثبيت الإعدادات. العينة صغيرة واصطناعية؛ لذا المقارنة دليل تعليمي على سلامة المسار وليست تقديرًا للإنتاج.

In [8]:
learning_rate = 2e-5 if TRAINING_MODE == "full_finetune" else 1e-4
trainable_parameters = [p for p in model.parameters() if p.requires_grad]
optimizer = AdamW(trainable_parameters, lr=learning_rate)
losses = []
train_steps = 0
epoch_history = []
best_validation_f1 = -1.0
best_epoch = 0
best_trainable_state = None

for epoch_index in range(NUM_EPOCHS):
    model.train()
    epoch_losses = []
    for batch in iter_batches(
        train_rows, BATCH_SIZE, shuffle=True, seed=SEED + epoch_index + 1
    ):
        optimizer.zero_grad(set_to_none=True)
        encoded = encode_batch(batch)
        output = model(**encoded)
        loss = output.loss
        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite training loss")
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable_parameters, max_norm=1.0)
        optimizer.step()
        loss_value = float(loss.detach().cpu())
        losses.append(loss_value)
        epoch_losses.append(loss_value)
        train_steps += 1

    epoch_predictions = predict(validation_rows)
    epoch_f1 = f1_score(
        [row["topic"] for row in validation_rows], epoch_predictions,
        labels=LABELS, average="macro", zero_division=0,
    )
    epoch_history.append({
        "epoch": epoch_index + 1,
        "mean_loss": float(np.mean(epoch_losses)),
        "validation_macro_f1": float(epoch_f1),
    })
    if epoch_f1 > best_validation_f1:
        best_validation_f1 = float(epoch_f1)
        best_epoch = epoch_index + 1
        if TRAINING_MODE == "partial_finetune_cpu":
            best_trainable_state = {
                name: parameter.detach().cpu().clone()
                for name, parameter in model.named_parameters()
                if parameter.requires_grad
            }
    print(
        f"epoch={epoch_index + 1:02d} mean_loss={np.mean(epoch_losses):.4f} "
        f"validation_macro_f1={epoch_f1:.4f}"
    )

if best_trainable_state is not None:
    with torch.no_grad():
        for name, parameter in model.named_parameters():
            if name in best_trainable_state:
                parameter.copy_(best_trainable_state[name].to(parameter.device))
    selected_epoch = best_epoch
else:
    selected_epoch = NUM_EPOCHS

assert train_steps >= 1 and all(math.isfinite(value) for value in losses)
assert any(parameter.requires_grad for parameter in model.base_model.parameters())
print("Transformer optimizer steps=PASS", {"steps": train_steps, "selected_epoch": selected_epoch})

epoch=01 mean_loss=1.3823 validation_macro_f1=0.3631
epoch=02 mean_loss=1.3404 validation_macro_f1=0.5500
Transformer optimizer steps=PASS {'steps': 12, 'selected_epoch': 2}


In [9]:
transformer_val_pred = predict(validation_rows)
transformer_val_f1 = f1_score(
    [row["topic"] for row in validation_rows],
    transformer_val_pred,
    labels=LABELS,
    average="macro",
    zero_division=0,
)
validation_delta = transformer_val_f1 - baseline_val_f1
print("Selected epoch:", selected_epoch)
print("Transformer validation macro-F1 (MEASURED_SMOKE):", round(transformer_val_f1, 4))
print("Validation delta vs TF-IDF baseline:", round(validation_delta, 4))
print("Validation predictions:", list(zip(
    [row["topic"] for row in validation_rows], transformer_val_pred
)))
assert 0.0 <= transformer_val_f1 <= 1.0

Selected epoch: 2
Transformer validation macro-F1 (MEASURED_SMOKE): 0.55
Validation delta vs TF-IDF baseline: -0.1167
Validation predictions: [('digital_service', 'transport'), ('digital_service', 'digital_service'), ('permit', 'permit'), ('permit', 'permit'), ('health', 'digital_service'), ('health', 'digital_service'), ('transport', 'transport'), ('transport', 'transport')]


### Error interpretation | تفسير خطأ واحد

في مجموعة Validation أخطأ نموذج Transformer في فئة `health` وتنبأ بـ `digital_service` بدلًا منها.

التفسير المحتمل هو أن مجموعة التدريب صغيرة ومصطنعة، لذلك لم يحصل النموذج على أمثلة كافية لتكوين حدود واضحة بين فئات الخدمات العامة والفئة الصحية.

هذا تفسير تعليمي للخطأ وليس إثباتًا لسبب قطعي. نحتاج بيانات أكثر وشرائح تقييم أوسع لتحليل الخطأ بثقة أعلى.

## 5) القياس النهائي المصغر

بعد تثبيت الإعدادات السابقة نقيس Test مرة واحدة. لا نغير الإعدادات بناء على هذه النتيجة ثم نعيد تسميتها Frozen Test.

In [10]:
test_truth = [row["topic"] for row in test_rows]
baseline_test_pred = baseline.predict([row["text"] for row in test_rows]).tolist()
transformer_test_pred = predict(test_rows)

results = {
    "result_type": "MEASURED_SMOKE",
    "data_source": DATA_SOURCE,
    "model_id": MODEL_ID,
    "device": str(DEVICE),
    "training_mode": TRAINING_MODE,
    "seed": SEED,
    "epochs_run": NUM_EPOCHS,
    "selected_epoch": selected_epoch,
    "train_steps": train_steps,
    "mean_train_loss": float(np.mean(losses)),
    "baseline_validation_macro_f1": float(baseline_val_f1),
    "transformer_validation_macro_f1": float(transformer_val_f1),
    "validation_delta_vs_baseline": float(validation_delta),
    "baseline_beaten_on_validation": bool(validation_delta > 0),
    "baseline_test_macro_f1": float(f1_score(
        test_truth, baseline_test_pred, labels=LABELS,
        average="macro", zero_division=0,
    )),
    "transformer_test_macro_f1": float(f1_score(
        test_truth, transformer_test_pred, labels=LABELS,
        average="macro", zero_division=0,
    )),
    "transformer_test_accuracy": float(accuracy_score(test_truth, transformer_test_pred)),
    "limitations": [
        "synthetic tiny dataset",
        "small validation set used for epoch selection",
        "not an estimate of production quality",
    ],
}
print(json.dumps(results, ensure_ascii=False, indent=2))
Path("day2_classification_metrics.json").write_text(
    json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8"
)

{
  "result_type": "MEASURED_SMOKE",
  "data_source": "github_course_file",
  "model_id": "distilbert/distilbert-base-multilingual-cased",
  "device": "cuda",
  "training_mode": "full_finetune",
  "seed": 42,
  "epochs_run": 2,
  "selected_epoch": 2,
  "train_steps": 12,
  "mean_train_loss": 1.3613132337729137,
  "baseline_validation_macro_f1": 0.6666666666666666,
  "transformer_validation_macro_f1": 0.55,
  "validation_delta_vs_baseline": -0.11666666666666659,
  "baseline_beaten_on_validation": false,
  "baseline_test_macro_f1": 0.7333333333333333,
  "transformer_test_macro_f1": 0.35,
  "transformer_test_accuracy": 0.375,
  "limitations": [
    "synthetic tiny dataset",
    "small validation set used for epoch selection",
    "not an estimate of production quality"
  ]
}


782

لا تستنتج أن Transformer «فشل» أو «نجح إنتاجيًا» من هذه العينة. سجل الفرق، ثم اكتب ما تحتاجه لتقدير موثوق: بيانات أكثر، عدة بذور، slices، وتقييم مجمد.

## مستويات التحدي

- **Core:** الخلايا السابقة فقط.
- **Explore:** على GPU قارن full fine-tuning بالمسار الجزئي، مع تثبيت البيانات والبذرة والـvalidation.
- **Distinction:** نفذ ثلاث بذور وسجل mean ± range؛ لا تستخدم Test لاختيار البذرة.

In [11]:
core_checks = {
    "split_isolation": split_report["group_overlap"] == 0,
    "baseline_valid": 0.0 <= baseline_val_f1 <= 1.0,
    "training_ran": train_steps >= 1,
    "transformer_weights_updated": any(
        parameter.requires_grad for parameter in model.base_model.parameters()
    ),
    "loss_is_finite": all(math.isfinite(value) for value in losses),
    "validation_valid": 0.0 <= transformer_val_f1 <= 1.0,
    "baseline_comparison_recorded": math.isfinite(validation_delta),
    "result_is_honest": results["result_type"] == "MEASURED_SMOKE",
    "bilingual_data": {"ar", "en"} <= {row["language"] for row in rows},
}
for name, passed in core_checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")
assert all(core_checks.values())
print("DAY2_NOTEBOOK3_CORE=PASS")

split_isolation: PASS
baseline_valid: PASS
training_ran: PASS
transformer_weights_updated: PASS
loss_is_finite: PASS
validation_valid: PASS
baseline_comparison_recorded: PASS
result_is_honest: PASS
bilingual_data: PASS
DAY2_NOTEBOOK3_CORE=PASS


In [12]:
# Day 2 — Sentiment classification evidence
# Independent TF-IDF baseline + Transformer head

SENTIMENT_LABELS = sorted({row["sentiment"] for row in rows})
sentiment_label2id = {
    label: index for index, label in enumerate(SENTIMENT_LABELS)
}
sentiment_id2label = {
    index: label for label, index in sentiment_label2id.items()
}

# -------------------------
# 1) TF-IDF baseline
# -------------------------

sentiment_baseline = make_pipeline(
    TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=1,
    ),
    LinearSVC(random_state=SEED),
)

sentiment_baseline.fit(
    [row["text"] for row in train_rows],
    [row["sentiment"] for row in train_rows],
)

sentiment_baseline_val_pred = sentiment_baseline.predict(
    [row["text"] for row in validation_rows]
)

sentiment_baseline_val_f1 = f1_score(
    [row["sentiment"] for row in validation_rows],
    sentiment_baseline_val_pred,
    labels=SENTIMENT_LABELS,
    average="macro",
    zero_division=0,
)

print(
    "Sentiment TF-IDF validation Macro-F1:",
    round(sentiment_baseline_val_f1, 4),
)

# -------------------------
# 2) Transformer
# -------------------------

sentiment_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=len(SENTIMENT_LABELS),
    label2id=sentiment_label2id,
    id2label=sentiment_id2label,
).to(DEVICE)

# Same training policy used by the notebook.
if DEVICE.type == "cpu":
    for parameter in sentiment_model.base_model.parameters():
        parameter.requires_grad = False

    for parameter in sentiment_model.base_model.transformer.layer[-1].parameters():
        parameter.requires_grad = True

sentiment_trainable = [
    p for p in sentiment_model.parameters()
    if p.requires_grad
]

sentiment_optimizer = AdamW(
    sentiment_trainable,
    lr=2e-5 if DEVICE.type == "cuda" else 1e-4,
)

def encode_sentiment_batch(batch):
    encoded = tokenizer(
        [row["text"] for row in batch],
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )

    encoded["labels"] = torch.tensor(
        [
            sentiment_label2id[row["sentiment"]]
            for row in batch
        ],
        dtype=torch.long,
    )

    return {
        key: value.to(DEVICE)
        for key, value in encoded.items()
    }


def predict_sentiment(examples):
    sentiment_model.eval()
    predictions = []

    with torch.no_grad():
        for batch in iter_batches(examples, BATCH_SIZE):
            encoded = encode_sentiment_batch(batch)
            logits = sentiment_model(**encoded).logits
            predictions.extend(
                logits.argmax(-1).cpu().tolist()
            )

    return [
        sentiment_id2label[index]
        for index in predictions
    ]


best_sentiment_f1 = -1.0
best_sentiment_epoch = 0
best_sentiment_state = None

for epoch_index in range(NUM_EPOCHS):
    sentiment_model.train()

    for batch in iter_batches(
        train_rows,
        BATCH_SIZE,
        shuffle=True,
        seed=SEED + epoch_index + 1,
    ):
        sentiment_optimizer.zero_grad(set_to_none=True)

        encoded = encode_sentiment_batch(batch)
        output = sentiment_model(**encoded)

        output.loss.backward()

        torch.nn.utils.clip_grad_norm_(
            sentiment_trainable,
            1.0,
        )

        sentiment_optimizer.step()

    val_pred = predict_sentiment(validation_rows)

    val_f1 = f1_score(
        [row["sentiment"] for row in validation_rows],
        val_pred,
        labels=SENTIMENT_LABELS,
        average="macro",
        zero_division=0,
    )

    print(
        f"epoch={epoch_index + 1:02d} "
        f"sentiment_validation_macro_f1={val_f1:.4f}"
    )

    if val_f1 > best_sentiment_f1:
        best_sentiment_f1 = float(val_f1)
        best_sentiment_epoch = epoch_index + 1

        best_sentiment_state = {
            name: parameter.detach().cpu().clone()
            for name, parameter in sentiment_model.named_parameters()
            if parameter.requires_grad
        }

# Restore best validation checkpoint.
with torch.no_grad():
    for name, parameter in sentiment_model.named_parameters():
        if name in best_sentiment_state:
            parameter.copy_(
                best_sentiment_state[name].to(parameter.device)
            )

sentiment_val_pred = predict_sentiment(validation_rows)

sentiment_transformer_val_f1 = f1_score(
    [row["sentiment"] for row in validation_rows],
    sentiment_val_pred,
    labels=SENTIMENT_LABELS,
    average="macro",
    zero_division=0,
)

sentiment_delta = (
    sentiment_transformer_val_f1
    - sentiment_baseline_val_f1
)

print()
print("=== SENTIMENT RESULTS ===")
print(
    "TF-IDF Macro-F1:",
    round(sentiment_baseline_val_f1, 4),
)
print(
    "Transformer Macro-F1:",
    round(sentiment_transformer_val_f1, 4),
)
print(
    "Delta:",
    round(sentiment_delta, 4),
)
print(
    "Selected epoch:",
    best_sentiment_epoch,
)

assert 0.0 <= sentiment_baseline_val_f1 <= 1.0
assert 0.0 <= sentiment_transformer_val_f1 <= 1.0

print("DAY2_SENTIMENT_HEAD=PASS")

Sentiment TF-IDF validation Macro-F1: 0.0741


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


epoch=01 sentiment_validation_macro_f1=0.1333
epoch=02 sentiment_validation_macro_f1=0.1333

=== SENTIMENT RESULTS ===
TF-IDF Macro-F1: 0.0741
Transformer Macro-F1: 0.1333
Delta: 0.0593
Selected epoch: 1
DAY2_SENTIMENT_HEAD=PASS


In [13]:
# Day 2 — Notebook 03
# Official Explore + Distinction
# Explore: full fine-tuning vs partial fine-tuning on the same data/seed/validation
# Distinction: 3 seeds, report mean ± range
# IMPORTANT: Test is not used for model/seed selection.

import copy

EXPLORE_SEED = 42
DISTINCTION_SEEDS = [7, 42, 2026]


def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def train_topic_once(seed, mode):
    assert mode in {"full", "partial"}

    set_all_seeds(seed)

    run_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID,
        num_labels=len(LABELS),
        label2id=label2id,
        id2label=id2label,
    ).to(DEVICE)

    if mode == "partial":
        for parameter in run_model.base_model.parameters():
            parameter.requires_grad = False

        for parameter in run_model.base_model.transformer.layer[-1].parameters():
            parameter.requires_grad = True

    trainable_parameters = [
        parameter
        for parameter in run_model.parameters()
        if parameter.requires_grad
    ]

    learning_rate = 2e-5 if mode == "full" else 1e-4

    run_optimizer = AdamW(
        trainable_parameters,
        lr=learning_rate,
    )

    def encode_run_batch(batch):
        encoded = tokenizer(
            [row["text"] for row in batch],
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )

        encoded["labels"] = torch.tensor(
            [label2id[row["topic"]] for row in batch],
            dtype=torch.long,
        )

        return {
            key: value.to(DEVICE)
            for key, value in encoded.items()
        }

    for epoch_index in range(NUM_EPOCHS):
        run_model.train()

        for batch in iter_batches(
            train_rows,
            BATCH_SIZE,
            shuffle=True,
            seed=seed + epoch_index + 1,
        ):
            run_optimizer.zero_grad(set_to_none=True)

            encoded = encode_run_batch(batch)
            output = run_model(**encoded)

            loss = output.loss
            assert torch.isfinite(loss)

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                trainable_parameters,
                max_norm=1.0,
            )

            run_optimizer.step()

    run_model.eval()
    predictions = []

    with torch.no_grad():
        for batch in iter_batches(validation_rows, BATCH_SIZE):
            encoded = tokenizer(
                [row["text"] for row in batch],
                padding=True,
                truncation=True,
                max_length=MAX_LENGTH,
                return_tensors="pt",
            )

            encoded = {
                key: value.to(DEVICE)
                for key, value in encoded.items()
            }

            logits = run_model(**encoded).logits

            predictions.extend(
                logits.argmax(-1).cpu().tolist()
            )

    predicted_labels = [
        id2label[index]
        for index in predictions
    ]

    score = f1_score(
        [row["topic"] for row in validation_rows],
        predicted_labels,
        labels=LABELS,
        average="macro",
        zero_division=0,
    )

    del run_model
    del run_optimizer

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return float(score)


# --------------------------------------------------
# Explore
# --------------------------------------------------

# Seed 42 full result already came from the official Run all.
explore_full_f1 = float(transformer_val_f1)

explore_partial_f1 = train_topic_once(
    EXPLORE_SEED,
    "partial",
)

print("=== EXPLORE: FULL vs PARTIAL ===")
print("Seed:", EXPLORE_SEED)
print("Full fine-tuning validation Macro-F1:", round(explore_full_f1, 4))
print("Partial fine-tuning validation Macro-F1:", round(explore_partial_f1, 4))
print(
    "Difference (full - partial):",
    round(explore_full_f1 - explore_partial_f1, 4),
)

print("DAY2_NOTEBOOK3_EXPLORE=PASS")


# --------------------------------------------------
# Distinction
# --------------------------------------------------

# Reuse the official seed-42 full result.
seed_scores = {
    42: explore_full_f1
}

for seed in [7, 2026]:
    seed_scores[seed] = train_topic_once(
        seed,
        "full",
    )

ordered_scores = [
    seed_scores[seed]
    for seed in DISTINCTION_SEEDS
]

mean_f1 = float(np.mean(ordered_scores))
range_f1 = float(
    max(ordered_scores) - min(ordered_scores)
)

print()
print("=== DISTINCTION: THREE SEEDS ===")

for seed in DISTINCTION_SEEDS:
    print(
        f"seed={seed}: "
        f"validation_macro_f1={seed_scores[seed]:.4f}"
    )

print("Mean validation Macro-F1:", round(mean_f1, 4))
print("Range:", round(range_f1, 4))
print(
    "Mean ± range:",
    f"{mean_f1:.4f} ± {range_f1:.4f}",
)

assert len(seed_scores) == 3
assert all(
    0.0 <= score <= 1.0
    for score in ordered_scores
)

print("TEST_USED_FOR_SELECTION=False")
print("DAY2_NOTEBOOK3_DISTINCTION=PASS")
print("DAY2_NOTEBOOK3_EXPLORE_DISTINCTION=PASS")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


=== EXPLORE: FULL vs PARTIAL ===
Seed: 42
Full fine-tuning validation Macro-F1: 0.55
Partial fine-tuning validation Macro-F1: 0.75
Difference (full - partial): -0.2
DAY2_NOTEBOOK3_EXPLORE=PASS


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



=== DISTINCTION: THREE SEEDS ===
seed=7: validation_macro_f1=0.1000
seed=42: validation_macro_f1=0.5500
seed=2026: validation_macro_f1=0.3429
Mean validation Macro-F1: 0.331
Range: 0.45
Mean ± range: 0.3310 ± 0.4500
TEST_USED_FOR_SELECTION=False
DAY2_NOTEBOOK3_DISTINCTION=PASS
DAY2_NOTEBOOK3_EXPLORE_DISTINCTION=PASS


## نقطة GitHub

1. احفظ نسخة notebook في مستودعك بالاسم نفسه.
2. لا ترفع Hugging Face cache أو model weights.
3. ارفع `day2_classification_metrics.json` ضمن `reports/` إذا لم يحتو بيانات شخصية.
4. حدّث `DECISIONS.md` بنوع التدريب والسبب وحدود العينة.
5. لا تعمل commit النهائي لليوم حتى تكمل دفتر NER وQA.

**التالي:** [مختبر NER وQA](04_ner_and_qa.ipynb).